In [5]:
import time
import pandas as pd #toolbox to work with dataframes
import numpy as np #toolbox to work with narrays
import matplotlib.pyplot as plt #toolbox to do plots
from sklearn.svm import SVC #load the support vector machine model functions
from sklearn.model_selection import train_test_split #load the function to split train and test sets
from sklearn import metrics # get the report
from sklearn.metrics import classification_report # get the report
from sklearn import preprocessing # normalize the features
from sklearn.preprocessing import MinMaxScaler # normalize the features
from sklearn.feature_selection import SelectKBest #load the feature selector model  
from sklearn.feature_selection import chi2 #feature selector algorithm
from sklearn.decomposition import PCA, FastICA, LatentDirichletAllocation, DictionaryLearning, FactorAnalysis, NMF
import warnings
warnings.filterwarnings('ignore')
#from tqdm.notebook import tqdm
from tqdm.notebook import tqdm
from sklearn.pipeline import Pipeline


from sklearn.preprocessing import FunctionTransformer, MinMaxScaler, StandardScaler


def normalize_Functions(method="zscore"):
    
    if method == "minmax":
        return MinMaxScaler()

    elif method == "zscore":
        return StandardScaler()




# Load the function to performe feature selection
from sklearn.feature_selection import SelectKBest
#load the feature selection algorithms
from sklearn.feature_selection import chi2
from sklearn.feature_selection import mutual_info_classif
from sklearn.feature_selection import f_regression
from sklearn.feature_selection import f_classif
from sklearn.feature_selection import mutual_info_regression
from sklearn.feature_selection import SelectPercentile
from sklearn.feature_selection import SelectFpr
from sklearn.feature_selection import SelectFdr
from sklearn.feature_selection import SelectFwe
from sklearn.feature_selection import GenericUnivariateSelect

# function to do feature selection
def feature_selector(type,i):
    if (type == 1):
#ANOVA F-value between label/feature for classification tasks.
        return SelectKBest(score_func = f_classif, k=i)
    elif(type == 2):
#Mutual information for a discrete target.
        return SelectKBest(score_func=mutual_info_classif, k=i)
    elif(type == 3):
    #Chi-squared stats of non-negative features for classification tasks.
        return SelectKBest(score_func=chi2, k=i)
    elif(type == 4):
#Select features based on an estimated false discovery rate.
        return  SelectKBest(score_func=SelectFdr, k=i)
    elif(type == 5):
#Select features based on family-wise error rate.
        return  SelectKBest(score_func=SelectFwe, k=i)



def classification_metrics(target, predicted_target):
  fpr, tpr, thresholds = roc_curve(target, predicted_target, pos_label=target.max())
  areaUC = auc(fpr, tpr)
  cf_mat = confusion_matrix(target, predicted_target)
  
  cf_mat= np.transpose(cf_mat)
  
  
  TP = np.diag(cf_mat)
  acc1 =np.sum(np.diag(cf_mat))
  FP = np.sum(cf_mat, axis=1) - TP
  FN = np.sum(cf_mat, axis=0) - TP
  TN = np.sum(cf_mat) - (TP + FP + FN)
  total = np.sum(cf_mat)

  class_weights = np.sum(cf_mat, axis=0)/np.sum(cf_mat)
  
  accuracy1 = acc1 / total
  precision1 = []
  recall1 = []
  specificity1 = []
  f1_score1 = []
  ppv = []
  npv = []
  
  reshapeSize = len(cf_mat)

  if reshapeSize > 2:
      
      for classNumb in range(reshapeSize):
          # Calculate Precision for each class
          if TP[classNumb] + FP[classNumb] == 0:
              precision1.append(0)
          else:
              precision1.append(TP[classNumb] / (TP[classNumb] + FP[classNumb]))
          
          # Calculate Recall for each class
          if TP[classNumb] + FN[classNumb] == 0:
              recall1.append(0)
          else:
              recall1.append(TP[classNumb] / (TP[classNumb] + FN[classNumb]))
              
          # Calculate Recall for each class
          if TN[classNumb] + FP[classNumb] == 0:
              specificity1.append(0)
          else:
              specificity1.append(TN[classNumb] / (TN[classNumb] + FP[classNumb]))
              
          # Calculate Recall for each class
          if TP[classNumb] + FP[classNumb] == 0:
              ppv.append(0)
          else:
              ppv.append(TP[classNumb] / (TP[classNumb] + FP[classNumb]))
              
          # Calculate Recall for each class
          if TN[classNumb] + FN[classNumb] == 0:
              npv.append(0)
          else:
              npv.append(TN[classNumb] / (TN[classNumb] + FN[classNumb]))
          
             
  else:
      
      # Calculate Precision for each class
      if TP[0] + FP[0] == 0:
          precision1.append(0)
      else:
          precision1.append(TP[0] / (TP[0] + FP[0]))
      
      # Calculate Recall for each class
      if TP[0] + FN[0] == 0:
          recall1.append(0)
      else:
          recall1.append(TP[0] / (TP[0] + FN[0]))
          
      # Calculate Recall for each class
      if TN[0] + FP[0] == 0:
          specificity1.append(0)
      else:
          specificity1.append(TN[0] / (TN[0] + FP[0]))
          
      # Calculate Recall for each class
      if TP[0] + FP[0] == 0:
          ppv.append(0)
      else:
          ppv.append(TP[0] / (TP[0] + FP[0]))
          
      # Calculate Recall for each class
      if TN[0] + FN[0] == 0:
          npv.append(0)
      else:
          npv.append(TN[0] / (TN[0] + FN[0]))
      
       
  precision1 = np.array(precision1)
  recall1 = np.array(recall1)
  specificity1 = np.array(specificity1)
  #f1_score1 = np.array(f1_score1)
  ppv = np.array(ppv)
  npv = np.array(npv)
   
   # Apply class weights
  weighted_accuracy = accuracy1
  weighted_precision = np.sum(precision1 * class_weights)
  weighted_recall = np.sum(recall1 * class_weights)
  weighted_specificity = np.sum(specificity1 * class_weights)
  #weighted_f1_score = np.sum(f1_score1 * class_weights)
  weighted_f1_score = 2 * weighted_precision * weighted_recall / (weighted_precision + weighted_recall)
  weighted_ppv = np.sum(ppv * class_weights)
  weighted_npv = np.sum(npv * class_weights)
  return fpr, tpr, cf_mat, weighted_accuracy, weighted_precision, weighted_recall, weighted_specificity, weighted_f1_score,weighted_ppv,weighted_npv, areaUC




# 3rd step - load and design the classifiers

from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC, SVC
from sklearn.ensemble import BaggingClassifier,ExtraTreesClassifier,RandomForestClassifier,AdaBoostClassifier
from sklearn.gaussian_process import GaussianProcessClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.gaussian_process.kernels import RBF
classifiers = [
    SVC(gamma='auto',probability=True),
    KNeighborsClassifier(),
    LogisticRegression(solver='lbfgs'),
    #BaggingClassifier(),
    #GaussianNB(),
    DecisionTreeClassifier(max_depth=5),
    AdaBoostClassifier(),
    #LinearDiscriminantAnalysis(),
    #QuadraticDiscriminantAnalysis(),
    LinearSVC(),
    #OneVsRestClassifier(LinearSVC(random_state=0)),
]

# Cross validation
from sklearn.metrics import make_scorer, f1_score, accuracy_score, precision_score, recall_score
from sklearn.model_selection import cross_validate,cross_val_score
from sklearn import model_selection
from sklearn.model_selection import StratifiedKFold,KFold
from sklearn.metrics import confusion_matrix
import matplotlib.patches as patches
from sklearn.metrics import roc_curve,auc

    
def classification(features, target):

    statResults = pd.DataFrame()
    #resultados = pd.DataFrame()
    
    nTprFpr = pd.DataFrame()
    
    cfm = pd.DataFrame()
    
    
    targetName = []
    
    featNumb = []
    algorithm = []
    accuracy = []
    roc_auc = []
    recall = []
    precision = []
    specificity = []
    f1 = []
    pPv = []
    nPv = []
    
    PCAVal = []
    
    
    for pcaValue in tqdm(range(1,31,1), desc= 'CombinationVal', leave = 0):
        
        for j in tqdm(range(30,features.shape[1],10), desc= 'Feature Selection', leave = 0):    
         
            splits = 10

            kfold = model_selection.StratifiedKFold(n_splits=splits, shuffle=False, random_state=None)
            
            for classifier in classifiers:
                
                pipeline = Pipeline([
                    ("scaler", normalize_Functions(method="minmax")),  # Step 1: Transforms data
                    ("feature_selection", SelectKBest(k=j+1)),  # Step 2: Transforms data
                    ("pca", PCA(n_components=pcaValue)),  # Step 3: Transforms data
                    ("classifier",classifier),  # Final Step: Learns patterns & predicts
                ])
                
                try:
                    
                    y_pred = model_selection.cross_val_predict(pipeline, features, target, cv=kfold)#,error_score=1)
                    
                    fpr, tpr, cf_mat, acc, prec, rec, spec, f1score, ppv, npv, areaUC = classification_metrics(target, y_pred)
                        
                    algorithm.append(classifier.__class__.__name__)
                    accuracy.append(acc)
                    recall.append(rec)
                    precision.append(prec)
                    specificity.append(spec)
                    f1.append(f1score)
                    pPv.append(ppv)
                    nPv.append(npv)
                    roc_auc.append(areaUC)
                    featNumb.append(j+1)
                    PCAVal.append(pcaValue)
                    
                    fprRest = np.ones(10000-len(fpr))
                    fprRest[:] = np.nan
                    fpr = np.concatenate((fpr,fprRest), axis=0)
                    nTprFpr['fprFS'+str(j+1)+'.PCA_'+str(pcaValue)+'.'+classifier.__class__.__name__]= fpr
                
                    tprRest = np.ones(10000-len(tpr))
                    tprRest[:] = np.nan
                    tpr = np.concatenate((tpr,tprRest), axis=0)
                    nTprFpr['tprFS'+str(j+1)+'.PCA_'+str(pcaValue)+'.'+classifier.__class__.__name__] = tpr
                
                    cf_mat = cf_mat.reshape(-1)
                    cf_matRest = np.ones(10000-len(cf_mat))
                    cf_matRest[:] = np.nan
                    cf_mat = np.concatenate((cf_mat,cf_matRest), axis=0)
                    cfm['cf_matFS'+str(j+1)+'.PCA_'+str(pcaValue)+'.'+classifier.__class__.__name__] = cf_mat
                
                    del tpr,fpr,areaUC,cf_mat,tprRest,fprRest,cf_matRest
                
                except Exception as e:
                    pass
            
       
     
    statResults['# of Features']  = featNumb    
    statResults['# PCA']  = PCAVal 
    statResults['Algorithm']  = algorithm    
    statResults['Accuracy']  = accuracy    
    statResults['Recall']  = recall 
    statResults['Precision']  = precision  
    statResults['Specificity']  = specificity    
    statResults['F1 Score']  = f1    
    statResults['PPV'] = pPv
    statResults['NPV'] = nPv
    statResults['Auc']  = roc_auc    
    
    return statResults, nTprFpr, cfm  
    
   



#%%
path = '/Users/pedroribeiro/Desktop/FtirNirClara/'



excelSheets = pd.ExcelFile(path+"analysis 1 - cópia.xlsx").sheet_names    


for Sheet in range(len(excelSheets)):
    database = pd.read_excel(path+'analysis 1 - cópia.xlsx',sheet_name = excelSheets[Sheet]) 
    
    databaseTarget = database.pop('Target')
    
    
    
    featuresNames = list(database.columns)
    
        
    
    statResults, nTprFpr, cfm  = classification(database, databaseTarget)
                    
                
    cfm.to_csv(path+'Results'+'/ConfMat_SheetName_'+excelSheets[Sheet]+'_.csv') 
    
    nTprFpr.to_csv(path+'Results'+'/rocAUC_SheetName_'+excelSheets[Sheet]+'_.csv') 
    
    
    #featsSelectedDF.to_csv(path+'Results'+'/Features_SheetName_'+excelSheets[Sheet]+'_.csv')  
               
    
    statResults.to_excel(path+'Results'+'/resultados_'+excelSheets[Sheet]+'_.xlsx') 
     

CombinationVal:   0%|          | 0/30 [00:00<?, ?it/s]

Feature Selection:   0%|          | 0/168 [00:00<?, ?it/s]

Feature Selection:   0%|          | 0/168 [00:00<?, ?it/s]

Feature Selection:   0%|          | 0/168 [00:00<?, ?it/s]

Feature Selection:   0%|          | 0/168 [00:00<?, ?it/s]

...

Feature Selection:   0%|          | 0/468 [00:00<?, ?it/s]

Feature Selection:   0%|          | 0/468 [00:00<?, ?it/s]

Feature Selection:   0%|          | 0/468 [00:00<?, ?it/s]

Feature Selection:   0%|          | 0/468 [00:00<?, ?it/s]

Feature Selection:   0%|          | 0/468 [00:00<?, ?it/s]

### Classifcation per regions

In [6]:
import time
import pandas as pd #toolbox to work with dataframes
import numpy as np #toolbox to work with narrays
import matplotlib.pyplot as plt #toolbox to do plots
from sklearn.svm import SVC #load the support vector machine model functions
from sklearn.model_selection import train_test_split #load the function to split train and test sets
from sklearn import metrics # get the report
from sklearn.metrics import classification_report # get the report
from sklearn import preprocessing # normalize the features
from sklearn.preprocessing import MinMaxScaler # normalize the features
from sklearn.feature_selection import SelectKBest #load the feature selector model  
from sklearn.feature_selection import chi2 #feature selector algorithm
from sklearn.decomposition import PCA, FastICA, LatentDirichletAllocation, DictionaryLearning, FactorAnalysis, NMF
import warnings
warnings.filterwarnings('ignore')
#from tqdm.notebook import tqdm
from tqdm.notebook import tqdm
from sklearn.pipeline import Pipeline


from sklearn.preprocessing import FunctionTransformer, MinMaxScaler, StandardScaler


def normalize_Functions(method="zscore"):
    
    if method == "minmax":
        return MinMaxScaler()

    elif method == "zscore":
        return StandardScaler()




# Load the function to performe feature selection
from sklearn.feature_selection import SelectKBest
#load the feature selection algorithms
from sklearn.feature_selection import chi2
from sklearn.feature_selection import mutual_info_classif
from sklearn.feature_selection import f_regression
from sklearn.feature_selection import f_classif
from sklearn.feature_selection import mutual_info_regression
from sklearn.feature_selection import SelectPercentile
from sklearn.feature_selection import SelectFpr
from sklearn.feature_selection import SelectFdr
from sklearn.feature_selection import SelectFwe
from sklearn.feature_selection import GenericUnivariateSelect

# function to do feature selection
def feature_selector(type,i):
    if (type == 1):
#ANOVA F-value between label/feature for classification tasks.
        return SelectKBest(score_func = f_classif, k=i)
    elif(type == 2):
#Mutual information for a discrete target.
        return SelectKBest(score_func=mutual_info_classif, k=i)
    elif(type == 3):
    #Chi-squared stats of non-negative features for classification tasks.
        return SelectKBest(score_func=chi2, k=i)
    elif(type == 4):
#Select features based on an estimated false discovery rate.
        return  SelectKBest(score_func=SelectFdr, k=i)
    elif(type == 5):
#Select features based on family-wise error rate.
        return  SelectKBest(score_func=SelectFwe, k=i)



def classification_metrics(target, predicted_target):
  fpr, tpr, thresholds = roc_curve(target, predicted_target, pos_label=target.max())
  areaUC = auc(fpr, tpr)
  cf_mat = confusion_matrix(target, predicted_target)
  
  cf_mat= np.transpose(cf_mat)
  
  
  TP = np.diag(cf_mat)
  acc1 =np.sum(np.diag(cf_mat))
  FP = np.sum(cf_mat, axis=1) - TP
  FN = np.sum(cf_mat, axis=0) - TP
  TN = np.sum(cf_mat) - (TP + FP + FN)
  total = np.sum(cf_mat)

  class_weights = np.sum(cf_mat, axis=0)/np.sum(cf_mat)
  
  accuracy1 = acc1 / total
  precision1 = []
  recall1 = []
  specificity1 = []
  f1_score1 = []
  ppv = []
  npv = []
  
  reshapeSize = len(cf_mat)

  if reshapeSize > 2:
      
      for classNumb in range(reshapeSize):
          # Calculate Precision for each class
          if TP[classNumb] + FP[classNumb] == 0:
              precision1.append(0)
          else:
              precision1.append(TP[classNumb] / (TP[classNumb] + FP[classNumb]))
          
          # Calculate Recall for each class
          if TP[classNumb] + FN[classNumb] == 0:
              recall1.append(0)
          else:
              recall1.append(TP[classNumb] / (TP[classNumb] + FN[classNumb]))
              
          # Calculate Recall for each class
          if TN[classNumb] + FP[classNumb] == 0:
              specificity1.append(0)
          else:
              specificity1.append(TN[classNumb] / (TN[classNumb] + FP[classNumb]))
              
          # Calculate Recall for each class
          if TP[classNumb] + FP[classNumb] == 0:
              ppv.append(0)
          else:
              ppv.append(TP[classNumb] / (TP[classNumb] + FP[classNumb]))
              
          # Calculate Recall for each class
          if TN[classNumb] + FN[classNumb] == 0:
              npv.append(0)
          else:
              npv.append(TN[classNumb] / (TN[classNumb] + FN[classNumb]))
          
             
  else:
      
      # Calculate Precision for each class
      if TP[0] + FP[0] == 0:
          precision1.append(0)
      else:
          precision1.append(TP[0] / (TP[0] + FP[0]))
      
      # Calculate Recall for each class
      if TP[0] + FN[0] == 0:
          recall1.append(0)
      else:
          recall1.append(TP[0] / (TP[0] + FN[0]))
          
      # Calculate Recall for each class
      if TN[0] + FP[0] == 0:
          specificity1.append(0)
      else:
          specificity1.append(TN[0] / (TN[0] + FP[0]))
          
      # Calculate Recall for each class
      if TP[0] + FP[0] == 0:
          ppv.append(0)
      else:
          ppv.append(TP[0] / (TP[0] + FP[0]))
          
      # Calculate Recall for each class
      if TN[0] + FN[0] == 0:
          npv.append(0)
      else:
          npv.append(TN[0] / (TN[0] + FN[0]))
      
       
  precision1 = np.array(precision1)
  recall1 = np.array(recall1)
  specificity1 = np.array(specificity1)
  #f1_score1 = np.array(f1_score1)
  ppv = np.array(ppv)
  npv = np.array(npv)
   
   # Apply class weights
  weighted_accuracy = accuracy1
  weighted_precision = np.sum(precision1 * class_weights)
  weighted_recall = np.sum(recall1 * class_weights)
  weighted_specificity = np.sum(specificity1 * class_weights)
  #weighted_f1_score = np.sum(f1_score1 * class_weights)
  weighted_f1_score = 2 * weighted_precision * weighted_recall / (weighted_precision + weighted_recall)
  weighted_ppv = np.sum(ppv * class_weights)
  weighted_npv = np.sum(npv * class_weights)
  return fpr, tpr, cf_mat, weighted_accuracy, weighted_precision, weighted_recall, weighted_specificity, weighted_f1_score,weighted_ppv,weighted_npv, areaUC




# 3rd step - load and design the classifiers

from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC, SVC
from sklearn.ensemble import BaggingClassifier,ExtraTreesClassifier,RandomForestClassifier,AdaBoostClassifier
from sklearn.gaussian_process import GaussianProcessClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.gaussian_process.kernels import RBF
classifiers = [
    SVC(gamma='auto',probability=True),
    KNeighborsClassifier(),
    LogisticRegression(solver='lbfgs'),
    #BaggingClassifier(),
    #GaussianNB(),
    DecisionTreeClassifier(max_depth=5),
    AdaBoostClassifier(),
    #LinearDiscriminantAnalysis(),
    #QuadraticDiscriminantAnalysis(),
    LinearSVC(),
    #OneVsRestClassifier(LinearSVC(random_state=0)),
]

# Cross validation
from sklearn.metrics import make_scorer, f1_score, accuracy_score, precision_score, recall_score
from sklearn.model_selection import cross_validate,cross_val_score
from sklearn import model_selection
from sklearn.model_selection import StratifiedKFold,KFold
from sklearn.metrics import confusion_matrix
import matplotlib.patches as patches
from sklearn.metrics import roc_curve,auc

    
def classification(features, target):

    statResults = pd.DataFrame()
    #resultados = pd.DataFrame()
    
    nTprFpr = pd.DataFrame()
    
    cfm = pd.DataFrame()
    
    
    targetName = []
    
    featNumb = []
    algorithm = []
    accuracy = []
    roc_auc = []
    recall = []
    precision = []
    specificity = []
    f1 = []
    pPv = []
    nPv = []
    
    PCAVal = []
    
    
    for pcaValue in tqdm(range(1,31,1), desc= 'CombinationVal', leave = 0):
        
        for j in tqdm(range(30,features.shape[1],10), desc= 'Feature Selection', leave = 0):    
         
            splits = 10

            kfold = model_selection.StratifiedKFold(n_splits=splits, shuffle=False, random_state=None)
            
            for classifier in classifiers:
                
                pipeline = Pipeline([
                    ("scaler", normalize_Functions(method="minmax")),  # Step 1: Transforms data
                    ("feature_selection", SelectKBest(k=j+1)),  # Step 2: Transforms data
                    ("pca", PCA(n_components=pcaValue)),  # Step 3: Transforms data
                    ("classifier",classifier),  # Final Step: Learns patterns & predicts
                ])
                
                try:
                    
                    y_pred = model_selection.cross_val_predict(pipeline, features, target, cv=kfold)#,error_score=1)
                    
                    fpr, tpr, cf_mat, acc, prec, rec, spec, f1score, ppv, npv, areaUC = classification_metrics(target, y_pred)
                        
                    algorithm.append(classifier.__class__.__name__)
                    accuracy.append(acc)
                    recall.append(rec)
                    precision.append(prec)
                    specificity.append(spec)
                    f1.append(f1score)
                    pPv.append(ppv)
                    nPv.append(npv)
                    roc_auc.append(areaUC)
                    featNumb.append(j+1)
                    PCAVal.append(pcaValue)
                    
                    fprRest = np.ones(10000-len(fpr))
                    fprRest[:] = np.nan
                    fpr = np.concatenate((fpr,fprRest), axis=0)
                    nTprFpr['fprFS'+str(j+1)+'.PCA_'+str(pcaValue)+'.'+classifier.__class__.__name__]= fpr
                
                    tprRest = np.ones(10000-len(tpr))
                    tprRest[:] = np.nan
                    tpr = np.concatenate((tpr,tprRest), axis=0)
                    nTprFpr['tprFS'+str(j+1)+'.PCA_'+str(pcaValue)+'.'+classifier.__class__.__name__] = tpr
                
                    cf_mat = cf_mat.reshape(-1)
                    cf_matRest = np.ones(10000-len(cf_mat))
                    cf_matRest[:] = np.nan
                    cf_mat = np.concatenate((cf_mat,cf_matRest), axis=0)
                    cfm['cf_matFS'+str(j+1)+'.PCA_'+str(pcaValue)+'.'+classifier.__class__.__name__] = cf_mat
                
                    del tpr,fpr,areaUC,cf_mat,tprRest,fprRest,cf_matRest
                
                except Exception as e:
                    pass
            
       
     
    statResults['# of Features']  = featNumb    
    statResults['# PCA']  = PCAVal 
    statResults['Algorithm']  = algorithm    
    statResults['Accuracy']  = accuracy    
    statResults['Recall']  = recall 
    statResults['Precision']  = precision  
    statResults['Specificity']  = specificity    
    statResults['F1 Score']  = f1    
    statResults['PPV'] = pPv
    statResults['NPV'] = nPv
    statResults['Auc']  = roc_auc    
    
    return statResults, nTprFpr, cfm  
    
   




#%%
path = '/Users/pedroribeiro/Desktop/FtirNirClara/'

corIndex = {'Azul' : [9920,7274],
            'Vermelho' : [7270,6314],
            'Verde' : [6310,5388],
            'Amarelo' : [5384,4926],
            'Roxo' : [4922,4072],
            'R1' : [3000,2800] ,
            'R2' : [1750,1500],
            'R3' : [1500,1200],
            'R4' : [1200,900]}

corIndexKeys = list(corIndex.keys())


for cores in tqdm(corIndexKeys, desc= 'Color', leave = 0):
    
    database = pd.read_excel(path+'analysis 1 - cópia.xlsx',sheet_name = 'Combined_spe') 
    
    databaseTarget = database.pop('Target')
    
    x = database.copy()
    x = x.iloc[:,x.columns.get_loc(corIndex[cores][0]):x.columns.get_loc(corIndex[cores][1])+1]
    
    featuresNames = list(database.columns)
    
        
    
    statResults, nTprFpr, cfm  = classification(x, databaseTarget)
                    
                
    cfm.to_csv(path+'Results'+'/ConfMat_SheetName_'+'Color_'+cores+'_.csv') 
    
    nTprFpr.to_csv(path+'Results'+'/rocAUC_SheetName_'+'Color_'+cores+'_.csv') 
              
    statResults.to_excel(path+'Results'+'/resultados_'+'Color_'+cores+'_.xlsx') 
     

Color:   0%|          | 0/9 [00:00<?, ?it/s]

CombinationVal:   0%|          | 0/30 [00:00<?, ?it/s]

Feature Selection:   0%|          | 0/130 [00:00<?, ?it/s]

Feature Selection:   0%|          | 0/130 [00:00<?, ?it/s]

Feature Selection:   0%|          | 0/130 [00:00<?, ?it/s]

...

Feature Selection:   0%|          | 0/13 [00:00<?, ?it/s]

Feature Selection:   0%|          | 0/13 [00:00<?, ?it/s]

Feature Selection:   0%|          | 0/13 [00:00<?, ?it/s]

Feature Selection:   0%|          | 0/13 [00:00<?, ?it/s]

Feature Selection:   0%|          | 0/13 [00:00<?, ?it/s]